# Phase 6 — Explainable AI

I inspect gradient-based explanations and distinguish model sensitivity from biological causality.

> **Research integrity:** Figures labeled synthetic are not clinical EEG observations. Run code locally in VS Code using the project `.venv` interpreter.

## 1. What input-gradient saliency measures

We differentiate one predicted class logit with respect to the input signal. The result is a *local sensitivity measure*. It is not proof that a neuron, frequency component, or clinical phenomenon caused the classification.

In [ ]:
import torch
from eeg_sleep.models import get_model
model=get_model("cnn").eval()
x=torch.randn(1,2,3000,requires_grad=True)
logits=model(x)
class_id=int(logits.argmax(dim=1))
logits[0,class_id].backward()
saliency=x.grad.detach().abs()
print("Raw input shape:",tuple(x.shape),"Gradient shape:",tuple(saliency.shape))

## 2. Visualize a gradient for an UNTRAINED model

This graph demonstrates the operation of autograd only. It is **not** an explanation of meaningful sleep-stage decisions.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,3))
ax.plot(saliency[0,0].numpy(),linewidth=.6)
ax.set(xlabel="Sample index",ylabel="Absolute input gradient",title="Untrained CNN: autograd demonstration only")
plt.show()

## 3. Explain a trained checkpoint (optional)

The CLI stores a PNG under `results/`. Only use checkpoints created by this project and never load unknown PyTorch checkpoints.

In [ ]:
from pathlib import Path
ckpt=Path("results/deep/cnn_checkpoint.pt")
p=Path("data/processed/synthetic_demo.npz")
if ckpt.exists() and p.exists():
    from eeg_sleep.explain import generate_saliency
    print(generate_saliency(p,ckpt,"results/deep/saliency.png"))
else: print("No locally trained CNN checkpoint; run the demo and CNN command first.")

## Exercises

1. What is the mathematical gradient being plotted? 2. Why can gradient magnitude be noisy? 3. Why should I test interpretation stability across subjects? 4. Why does saliency not constitute causal evidence?